In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.1 MB/s eta 0:00:00


In [1]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.3 MB/s eta 0:00:00


In [3]:
import json
import random
import re

DATA_FILE = "actions_training_500.json"
random.seed(42)


def load_examples(path):
    text = open(path, encoding="utf-8").read().strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError:  # JSON-lines file
        data = [json.loads(line) for line in text.splitlines() if line.strip()]

    if isinstance(data, dict):  # e.g. {"examples": [...]}
        data = next(v for v in data.values() if isinstance(v, list))

    return data


def clean(example):
    params = {
        key: value
        for key, value in (example["parameters"] or {}).items()
        if value not in (None, "", [], {})
    }

    return {
        "utterance": example["utterance"].strip(),
        "action": example["action"],
        "parameters": params,
    }


examples = [clean(e) for e in load_examples(DATA_FILE)]

print(len(examples), "examples")
print(examples[0])

500 examples
{'utterance': 'Get calendar events for next Monday between 14:30 and 17:00.', 'action': 'get_calendar_events', 'parameters': {'date': 'next Monday', 'start_time': '14:30', 'end_time': '17:00'}}


In [4]:
random.seed(42)
random.shuffle(examples)
cut = int(len(examples) * 0.8)
train_orig, val_orig = examples[:cut], examples[cut:]   # split BEFORE augmenting

# ---- synonym variants of training examples ----
SYNONYMS = [
    ("support ticket", ["service request", "helpdesk ticket"], {"create_support_ticket"}),
    ("task", ["to-do"], {"create_task", "update_task", "delete_task"}),
    ("meeting", ["call", "sync"], {"schedule_meeting"}),
    ("create", ["add", "make", "set up"], None),
    ("delete", ["remove"], {"delete_task"}),
    ("schedule", ["set up", "arrange"], {"schedule_meeting"}),
]


def match_case(source, replacement):
    if source[:1].isupper():
        return replacement[0].upper() + replacement[1:]
    return replacement


def synonym_variants(example):
    variants = []
    blob = json.dumps(example["parameters"]).lower()

    for phrase, replacements, only in SYNONYMS:

        if only and example["action"] not in only:
            continue

        match = re.search(rf"\b{re.escape(phrase)}\b", example["utterance"], re.IGNORECASE)

        # skip if the phrase is part of a parameter value (we'd desync the answer)
        if not match or phrase in blob:
            continue

        for replacement in random.sample(replacements, min(2, len(replacements))):
            text = (
                example["utterance"][:match.start()]
                + match_case(match.group(0), replacement)
                + example["utterance"][match.end():]
            )
            variants.append({**example, "utterance": text})

    return variants


all_variants = [v for e in train_orig for v in synonym_variants(e)]
synonym_examples = random.sample(all_variants, min(len(all_variants), 250))

# ---- bare requests: action is clear, nothing else said ----
BARE = {
    "create_task": [
        "create a task", "add a new task", "I need to make a task",
        "new to-do please", "can you add a task for me", "make a task",
    ],
    "send_message": [
        "send a message", "I want to message someone",
        "can you send a message for me", "drop a message", "send a quick message",
    ],
    "schedule_meeting": [
        "schedule a meeting", "set up a meeting", "I need to book a meeting",
        "arrange a call", "can you schedule a call for me", "let's set up a sync",
    ],
    "create_calendar_event": [
        "create a calendar event", "add an event to my calendar",
        "put something on my calendar", "add an appointment", "make a calendar entry",
    ],
    "search_documents": [
        "search my documents", "find a document", "look for a file",
        "I need to find a file", "can you search my files",
    ],
    "send_email": [
        "send an email", "I need to email someone", "write an email",
        "can you send a mail for me", "compose an email",
    ],
    "update_task": [
        "update a task", "edit a task", "I want to change a task",
        "modify an existing task", "change one of my tasks",
    ],
    "delete_task": [
        "delete a task", "remove a task", "I want to delete a task",
        "get rid of a task", "can you remove a task",
    ],
    "get_calendar_events": [
        "what's on my calendar", "show my calendar", "check my calendar",
        "what do I have scheduled", "show my upcoming events",
    ],
    "create_support_ticket": [
        "create a support ticket", "create a service request", "raise a support ticket",
        "raise a ticket", "log an issue", "open a support ticket",
        "submit a service request", "I need to file a service request",
    ],
}

bare_examples = [
    {"utterance": text, "action": action, "parameters": {}}
    for action, texts in BARE.items()
    for text in texts
]

# ---- not an action: chit-chat, questions, out-of-scope ----
NOT_AN_ACTION = [
    "hi", "hello", "hey there", "good morning", "good evening", "thanks",
    "thank you so much", "ok", "cool", "nice", "bye", "you are great",
    "how are you", "who are you", "what can you do", "help",
    "what is a service request", "what does a support ticket mean",
    "explain what a calendar event is", "what is a task", "define email",
    "what's the meaning of meeting", "how does a calendar work",
    "what's the weather today", "what's the weather in Paris",
    "tell me a joke", "who won the world cup", "what time is it in Tokyo",
    "what's 15 times 12", "translate hello to French", "play some music",
    "book me a flight to Delhi", "order a pizza", "turn off the lights",
    "what's the capital of France", "write me a poem", "recommend a movie",
    "what is machine learning", "how tall is Mount Everest",
    "can you tell me about yourself", "I'm bored", "I'm tired today",
    "that was helpful", "never mind", "no", "yes", "maybe later",
    "I had a meeting yesterday", "the task was really hard",
    "my email is slow today", "the calendar app looks nice",
    "we talked about the ticket earlier", "what is the difference between a task and a ticket",
    "why do we have meetings", "is email better than chat",
    "tell me about your day", "sing me a song", "what is your name",
]

negative_examples = [
    {"utterance": text, "action": None, "parameters": {}} for text in NOT_AN_ACTION
]

# ---- put it together ----
synthetic = bare_examples + negative_examples
random.shuffle(synthetic)
synthetic_cut = int(len(synthetic) * 0.8)

train_all = train_orig + synonym_examples + synthetic[:synthetic_cut]
val_all = val_orig + synthetic[synthetic_cut:]
random.shuffle(train_all)

print(f"train: {len(train_all)}   val: {len(val_all)}")

train: 740   val: 123


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
from datasets import Dataset

MAX_LENGTH = 512


def build_prompt(user_text):
    return f"""### Instruction:
Determine the action and parameters required for this user request.

### User:
{user_text}

### Response:
"""


def build_answer(example):
    return json.dumps(
        {"action": example["action"], "parameters": example["parameters"]},
        ensure_ascii=False,
    )


def tokenize(example):
    prompt = build_prompt(example["utterance"])
    answer = build_answer(example) + tokenizer.eos_token

    prompt_length = len(tokenizer(prompt)["input_ids"])

    encoded = tokenizer(prompt + answer, truncation=True, max_length=MAX_LENGTH)

    labels = list(encoded["input_ids"])
    labels[:prompt_length] = [-100] * prompt_length  # don't learn the prompt

    encoded["labels"] = labels
    return dict(encoded)


tokenized_train = Dataset.from_list([tokenize(e) for e in train_all])
tokenized_val = Dataset.from_list([tokenize(e) for e in val_all])

# sanity check: this must print the JSON followed by the end-of-sequence token
row = tokenized_train[0]
print(repr(tokenizer.decode([t for t, l in zip(row["input_ids"], row["labels"]) if l != -100])))

'{"action": "schedule_meeting", "parameters": {"title": "Update pricing sheet", "participants": "Priya", "date": "tomorrow", "start_time": "11:00", "duration": "45 minutes"}}<|im_end|>'


In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="/working/action-model-v2",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.009130,0.107686
2,0.001680,0.002643
3,0.000561,0.004581
4,0.000274,0.002195


TrainOutput(global_step=188, training_loss=0.03919612905233504, metrics={'train_runtime': 186.4943, 'train_samples_per_second': 15.872, 'train_steps_per_second': 1.008, 'total_flos': 665043379522560.0, 'train_loss': 0.03919612905233504, 'epoch': 4.0})

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [10]:
import torch
from transformers import StoppingCriteria, StoppingCriteriaList


class StopAfterJson(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(input_ids[0][self.prompt_length:], skip_special_tokens=True)
        start = text.find("{")
        if start == -1:
            return False
        try:
            json.JSONDecoder().raw_decode(text[start:])
            return True
        except ValueError:
            return False


def predict(utterance):
    prompt = build_prompt(utterance)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    n = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=StoppingCriteriaList([StopAfterJson(tokenizer, n)]),
        )

    text = tokenizer.decode(output[0][n:], skip_special_tokens=True)
    start = text.find("{")

    if start == -1:
        return None

    try:
        return json.JSONDecoder().raw_decode(text[start:])[0]
    except ValueError:
        return None


def norm(value):
    if isinstance(value, str):
        return value.strip().lower()
    if isinstance(value, list):
        return [norm(v) for v in value]
    if isinstance(value, dict):
        return {k: norm(v) for k, v in value.items() if v not in (None, "", [], {})}
    return value


model.eval()

stats = {}
failures = []

for example in val_all:
    prediction = predict(example["utterance"]) or {}

    name = example["action"] or "(no action)"
    stat = stats.setdefault(name, {"n": 0, "action": 0, "params": 0})
    stat["n"] += 1

    action_ok = prediction.get("action") == example["action"]
    params_ok = action_ok and norm(prediction.get("parameters") or {}) == norm(example["parameters"])

    stat["action"] += action_ok
    stat["params"] += params_ok

    if not params_ok:
        failures.append((example, prediction))

total = sum(s["n"] for s in stats.values())
print(f"\nACTION correct:      {sum(s['action'] for s in stats.values())}/{total}")
print(f"ACTION+PARAMS exact: {sum(s['params'] for s in stats.values())}/{total}\n")

for name, s in sorted(stats.items()):
    print(f"{name:26s} n={s['n']:3d}  action={s['action']:3d}  exact={s['params']:3d}")

print("\nFirst failures:")
for example, prediction in failures[:10]:
    print("-", example["utterance"])
    print("    expected:", build_answer(example))
    print("    got:     ", json.dumps(prediction, ensure_ascii=False))

print()
for text in [
    "create a service request",
    "raise a ticket for the broken login page",
    "hi",
    "what is a service request",
    "Create a high priority task to review the Q4 financial report and assign it to Rahul.",
    "schedule a meeting with Rahul tomorrow at 10am for 30 minutes",
    "send an email to sam",
]:
    print(text, "->", json.dumps(predict(text), ensure_ascii=False))


ACTION correct:      122/123
ACTION+PARAMS exact: 122/123

(no action)                n= 13  action= 13  exact= 13
create_calendar_event      n= 10  action= 10  exact= 10
create_support_ticket      n=  9  action=  9  exact=  9
create_task                n= 12  action= 12  exact= 12
delete_task                n= 16  action= 16  exact= 16
get_calendar_events        n= 11  action= 10  exact= 10
schedule_meeting           n= 14  action= 14  exact= 14
search_documents           n=  4  action=  4  exact=  4
send_email                 n= 13  action= 13  exact= 13
send_message               n=  9  action=  9  exact=  9
update_task                n= 12  action= 12  exact= 12

First failures:
- what's on my calendar
    expected: {"action": "get_calendar_events", "parameters": {}}
    got:      {"action": null, "parameters": {}}

create a service request -> {"action": "create_support_ticket", "parameters": {}}
raise a ticket for the broken login page -> {"action": "create_support_ticket", "para

In [13]:
model.save_pretrained("/working/action-model-v2")
tokenizer.save_pretrained("/working/action-model-v2")

('/working/action-model-v2/tokenizer_config.json',
 '/working/action-model-v2/chat_template.jinja',
 '/working/action-model-v2/tokenizer.json')

In [14]:
!zip -r /working/action-model-v2.zip /working/action-model-v2

updating: working/action-model-v2/ (stored 0%)
updating: working/action-model-v2/checkpoint-141/ (stored 0%)
updating: working/action-model-v2/checkpoint-141/rng_state.pth (deflated 26%)
updating: working/action-model-v2/checkpoint-141/scaler.pt (deflated 64%)
updating: working/action-model-v2/checkpoint-141/trainer_state.json (deflated 70%)
updating: working/action-model-v2/checkpoint-141/optimizer.pt (deflated 8%)
updating: working/action-model-v2/checkpoint-141/README.md (deflated 65%)
updating: working/action-model-v2/checkpoint-141/chat_template.jinja (deflated 71%)
updating: working/action-model-v2/checkpoint-141/tokenizer_config.json (deflated 59%)
updating: working/action-model-v2/checkpoint-141/training_args.bin (deflated 53%)
updating: working/action-model-v2/checkpoint-141/tokenizer.json (deflated 81%)
updating: working/action-model-v2/checkpoint-141/scheduler.pt (deflated 61%)
updating: working/action-model-v2/checkpoint-141/adapter_model.safetensors (deflated 7%)
updating:

In [ ]:
!zip -r /working/action-model.zip /working/action-model

  adding: working/action-model/ (stored 0%)
  adding: working/action-model/checkpoint-100/ (stored 0%)
  adding: working/action-model/checkpoint-100/rng_state.pth (deflated 26%)
  adding: working/action-model/checkpoint-100/scaler.pt (deflated 64%)
  adding: working/action-model/checkpoint-100/trainer_state.json (deflated 71%)
  adding: working/action-model/checkpoint-100/optimizer.pt (deflated 8%)
  adding: working/action-model/checkpoint-100/README.md (deflated 65%)
  adding: working/action-model/checkpoint-100/chat_template.jinja (deflated 71%)
  adding: working/action-model/checkpoint-100/tokenizer_config.json (deflated 60%)
  adding: working/action-model/checkpoint-100/training_args.bin (deflated 54%)
  adding: working/action-model/checkpoint-100/tokenizer.json (deflated 81%)
  adding: working/action-model/checkpoint-100/scheduler.pt (deflated 61%)
  adding: working/action-model/checkpoint-100/adapter_model.safetensors (deflated 8%)
  adding: working/action-model/checkpoint-100/ad